In [1]:
%matplotlib widget
import numpy as np
import xarray as xr
import csv
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.widgets import RectangleSelector

In [2]:
def create_long_lat(dx, nx, ny, beta_rot):
    r_earth = 6370.0 # radius of the Earth
    beta    = beta_rot   # to rotate the mask with respect to Greenwich
                     # when beta = 0, x and x1 axis are aligned with Greenwich
    if int(80 % dx)==0:
        dx_pole = 2480 + (80 - (dx / 2))  # distance of pole from tracer point of cell 0,0
        dy_pole = 2240 + (80 - (dx / 2)) 
    elif int(32 % dx)==0:
        dx_pole = 2464 + (32 - (dx / 2))  # distance of pole from tracer point of cell 0,0
        dy_pole = 2208 + (32 - (dx / 2)) 
    
    deg2rad = np.pi / 180
    rad2deg = 180 / np.pi
          
    latmin  = 90.0
    ilatmin = 0
    jlatmin = 0
    r1max   = 0

    lat = np.zeros((nx, ny))
    long = np.zeros((nx, ny))

    ii = np.arange(0, nx)
    jj = np.arange(0, ny)
    j, i = np.meshgrid(jj, ii)
    #---------- position on the stereographic plane -------------------------------
    x1 = i * dx - dx_pole
    y1 = j * dx - dy_pole
    r1 = np.sqrt(x1**2 + y1**2)
    if np.max(r1) > r1max:
        r1max = np.max(r1)
    #---------- angle of the cone -------------------------------------------------
    tanteta  = r1 / (2 * r_earth)
    #----------- short radius on the sphere ---------------------------------------
    r = 2 * r_earth * tanteta / (1 + tanteta**2)
    #----------- get the latitude and longitude -----------------------------------
    lat = np.arccos(r / r_earth) * rad2deg
    if np.min(lat < latmin):
        latmin = np.min(lat)
        ilatmin = i[np.unravel_index(lat.argmin(), lat.shape)]
        jlatmin = j[np.unravel_index(lat.argmin(), lat.shape)]
    x1 = np.where(((x1 >= 0.0) & (x1 < 1e-8)), 1e-8, x1)
    long = np.where(((x1 >= 0.0) & (y1 >= 0)), np.arctan(y1 / x1) * rad2deg + beta, long)
    long = np.where(((x1 >= 0.0) & (y1 < 0)), np.arctan(y1 / x1) * rad2deg + beta + 360.0, long)
    long = np.where((x1 < 0.0), np.arctan(y1 / x1) * rad2deg + beta + 180.0, long)
    long = np.where((long >= 360.0), long - 360.0, long)
    return long, lat

In [3]:
dx = 32.0
if dx == 80.0:
    nx = 64 # x-dim of the domain
    ny = 54 # y-dim of the domain
elif dx == 40.0:
    nx = 128
    ny = 108
elif dx == 20.0:
    nx = 256
    ny = 216
elif dx == 10.0:
    nx = 512
    ny = 432
elif dx == 5.0:
    nx = 1024
    ny = 864
elif dx == 2.5:
    nx = 2048
    ny = 1728
elif dx == 1.25:
    nx = 4096
    ny = 3456  
elif dx == 32.0:
    nx = 160
    ny = 135
elif dx == 16.0:
    nx = 320
    ny = 270
elif dx == 8.0:
    nx = 640
    ny = 540
elif dx == 4.0:
    nx = 1280
    ny = 1080
elif dx == 2.0:
    nx = 2560
    ny = 2160
elif dx == 1.0:
    nx = 5120
    ny = 4320

In [4]:
long, lat = create_long_lat(dx, nx+2, ny+2, 32.0)

In [5]:
maskC2 = xr.DataArray(np.zeros((nx+2, ny+2), dtype=int), 
                     dims={"x": np.arange(0, nx+2), "y": np.arange(0, ny+2)}, 
                     coords={"lon": (["x", "y"], long), "lat": (["x", "y"], lat)})

In [6]:
# get bathymetry, interpolate it to the grid
bathy = xr.open_mfdataset("/home/jan/Data/SIM_stuff/ETOPO_2022_v1_30s_N90W180_surface.nc").isel(lat=slice(18000, None))
bathy["lon"] = xr.where(bathy.lon>=0, bathy.lon, bathy.lon+360)
bathy["z"] *= -1

In [7]:
bathy_ongrid = bathy.interp(lon=maskC2.lon, lat=maskC2.lat, method="linear").compute()
bathy_ongrid = bathy_ongrid.where(bathy_ongrid.z>0, -10.0)
bathy_ongrid = bathy_ongrid.where(((bathy_ongrid.z>=5) | (bathy_ongrid.z==-10.0)), 5.0)
# make the first row and column the same as the second, last the same as second-last
# to account for the fact that they are halo points
bathy_ongrid["z"].data[0, 2:-2] = bathy_ongrid["z"].data[1, 2:-2]
bathy_ongrid["z"].data[-1, 2:-2] = bathy_ongrid["z"].data[-1, 2:-2]
bathy_ongrid["z"].data[2:-2, 0] = bathy_ongrid["z"].data[2:-2, 1]
bathy_ongrid["z"].data[2:-2, -1] = bathy_ongrid["z"].data[2:-2, -1]
# we ignore three of the corners because they are completely on land
# the corner in the Atlantic will be set to water, even though there is a
# very small portion of the Faroe Islands there
bathy_ongrid["z"].data[-20:, 0:20] = bathy_ongrid.z[-20:, 0:20].where(bathy_ongrid.z[-20:, 0:20]>5, 5)
# we also need to make sure there is no open boundary in Russia
bathy_ongrid["z"].data[:, -2::] = -10

In [8]:
save_bathy = False
if save_bathy:
    l = " " + "\n ".join(["\t".join(row.astype(str))  for row in bathy_ongrid.z.values])
    with open("/home/jan/Data/SIM_stuff/bathymetry_dx" + f"{dx:.2f}" + "_nx" + str(nx) + "_ny" + str(ny) + ".dat", "w") as m:
        m.write(l)

In [9]:
maskC = xr.where(bathy_ongrid.z[1:-1, 1:-1]>0, 1, 0).compute()

In [10]:
maskij = maskC[1:nx-1, 1:ny-1].values
maski0j = maskC[:, 1:ny-1].values
maskij0 = maskC[1:nx-1, :].values
maskimj = maskC[0:nx-2, 1:ny-1].values
maskimj0 = maskC[0:nx-2, :].values
maskipj = maskC[2:nx, 1:ny-1].values
maskipj0 = maskC[2:nx, :].values
maskijm = maskC[1:nx-1, 0:ny-2].values
maski0jm = maskC[:, 0:ny-2].values
maskijp = maskC[1:nx-1, 2:ny].values
maski0jp = maskC[:, 2:ny].values
maskimjm = maskC[0:nx-2, 0:ny-2].values
maskimjp = maskC[0:nx-2, 2:ny].values
maskipjp = maskC[2:nx, 2:ny].values
maskipjm = maskC[2:nx, 0:ny-2].values

In [11]:
# do everything for the "real" grid, then add the halo-points
for k in range(0, 10):   # sweep the grid 10 times for a better polish!
    maskC[1:nx-1, 1:ny-1] = np.where((maskij==1) & ((maskimj==0) & (maskipj==0)), 0, maskC[1:nx-1, 1:ny-1])
    maskC[1:nx-1, 1:ny-1] = np.where((maskij==1) & ((maskijm==0) & (maskijp==0)), 0, maskC[1:nx-1, 1:ny-1])
    maskC[1:nx-1, 1:ny-1] = np.where((maskij==1) & (((maskimjm + maskipjp)==0) & ((maskimjp + maskipjm)==2)), 0, maskC[1:nx-1, 1:ny-1])
    maskC[1:nx-1, 1:ny-1] = np.where((maskij==1) & (((maskimjp + maskipjm)==0) & ((maskimjm + maskipjp)==2)), 0, maskC[1:nx-1, 1:ny-1])
    #-------------- remove islands with 1 grid cell -------------------------------
    maskC[1:nx-1, 1:ny-1] = np.where(((maskij==0) & ((maskimjm + maskijm + maskipjm + maskimj + maskipj + maskimjp + maskijp + maskipjp)==8)), 1, maskC[1:nx-1, 1:ny-1])
    # Western boundary of the domain
    maskC[0, 1:ny-1] = np.where((maski0j[0, :]==1) & ((maskij[0, :]==0) | ((maski0jm[0, :]==0) & (maski0jp[0, :]==0))), 0, maskC[0, 1:ny-1])
    maskC[0, 1:ny-1] = np.where((maski0j[0, :]!=1) & ((maskij[0, :]==1) & ((maski0jm[0, :]==1) & (maski0jp[0, :]==1))), 1, maskC[0, 1:ny-1])
    # Eastern boundary of the domain
    maskC[-1, 1:ny-1] = np.where((maski0j[-1, :]==1) & ((maskij[-1, :]==0) | ((maski0jm[-1, :]==0) & (maski0jp[-1, :]==0))), 0, maskC[-1, 1:ny-1])
    maskC[-1, 1:ny-1] = np.where((maski0j[-1, :]!=1) & ((maskij[-1, :]==1) & ((maski0jm[-1, :]==1) & (maski0jp[-1, :]==1))), 1, maskC[-1, 1:ny-1])
    # Southern boundary of the domain
    maskC[1:nx-1, 0] = np.where((maskij0[:, 0]==1) & ((maskij[:, 0]==0) | ((maskimj0[:, 0]==0) & (maskipj0[:, 0]==0))), 0, maskC[1:nx-1, 0])
    maskC[1:nx-1, 0] = np.where((maskij0[:, 0]!=1) & ((maskimj0[:, 0]==1) & ((maskipj0[:, 0]==1) & (maskij[:, 0]==1))), 1, maskC[1:nx-1, 0])
    # Northern boundary of the domain
    maskC[1:nx-1, -1] = np.where((maskij0[:, -1]==1) & ((maskij[:, -1]==0) | ((maskimj0[:, -1]==0) & (maskipj0[:, -1]==0))), 0, maskC[1:nx-1, -1])
    maskC[1:nx-1, -1] = np.where((maskij0[:, -1]!=1) & ((maskimj0[:, -1]==1) & ((maskipj0[:, -1]==1) & (maskij[:, -1]==1))), 1, maskC[1:nx-1, -1])
    #---------- the four corners ------------------------------------------------- 
    if ((maskC[0, 0] == 1) & ((maskC[1, 0] == 0) | (maskC[0, 1] == 0))):
         maskC[0, 0] = 0
    if ((maskC[nx-1, 0] == 1) & ((maskC[nx-2, 0] == 0) | (maskC[nx-1, 1] == 0))):
        maskC[nx-1, 0] = 0
    if ((maskC[0, ny-1] == 1) & ((maskC[0, ny-2] == 0) | (maskC[1, ny-1] == 0))):
        maskC[0, ny-1] = 0
    if ((maskC[nx-1, ny-1] == 1) & ((maskC[nx-1, ny-2] == 0) | (maskC[nx-2, ny-1] == 0))):
        maskC[nx-1, ny-1] = 0
    #-------------- removes four grid cells square lakes --------------------------
    maskC[2:nx-2, 2:ny-2] = np.where((((maskij[1:-1, 1:-1] + maskipj[1:-1, 1:-1] + maskijp[1:-1, 1:-1] + maskipjp[1:-1, 1:-1])==4) 
                                      & ((maskijm[1:-1, 1:-1]  + maskipjm[1:-1, 1:-1]  + maskimj[1:-1, 1:-1]  + maskipj[2::, 1:-1]  
                                          + maskimjp[1:-1, 1:-1]  + maskipjp[2::, 1:-1] + maskijp[1:-1, 2::] + maskipjp[1:-1, 2::])==0)), 0, maskC[2:nx-2, 2:ny-2])
    maskC[3:nx-1, 2:ny-2] = np.where((((maskij[1:-1, 1:-1] + maskipj[1:-1, 1:-1] + maskijp[1:-1, 1:-1] + maskipjp[1:-1, 1:-1])==4) 
                                      & ((maskijm[1:-1, 1:-1]  + maskipjm[1:-1, 1:-1]  + maskimj[1:-1, 1:-1]  + maskipj[2::, 1:-1]  
                                          + maskimjp[1:-1, 1:-1]  + maskipjp[2::, 1:-1] + maskijp[1:-1, 2::] + maskipjp[1:-1, 2::])==0)), 0, maskC[3:nx-1, 2:ny-2])
    maskC[2:nx-2, 3:ny-1] = np.where((((maskij[1:-1, 1:-1] + maskipj[1:-1, 1:-1] + maskijp[1:-1, 1:-1] + maskipjp[1:-1, 1:-1])==4) 
                                      & ((maskijm[1:-1, 1:-1]  + maskipjm[1:-1, 1:-1]  + maskimj[1:-1, 1:-1]  + maskipj[2::, 1:-1]  
                                          + maskimjp[1:-1, 1:-1]  + maskipjp[2::, 1:-1] + maskijp[1:-1, 2::] + maskipjp[1:-1, 2::])==0)), 0, maskC[2:nx-2, 3:ny-1])
    maskC[3:nx-1, 3:ny-1] = np.where((((maskij[1:-1, 1:-1] + maskipj[1:-1, 1:-1] + maskijp[1:-1, 1:-1] + maskipjp[1:-1, 1:-1])==4) 
                                      & ((maskijm[1:-1, 1:-1]  + maskipjm[1:-1, 1:-1]  + maskimj[1:-1, 1:-1]  + maskipj[2::, 1:-1]  
                                          + maskimjp[1:-1, 1:-1]  + maskipjp[2::, 1:-1] + maskijp[1:-1, 2::] + maskipjp[1:-1, 2::])==0)), 0, maskC[3:nx-1, 3:ny-1])

# now adding the halos
maskC2.data[1:-1, 1:-1] = maskC
maskC2.data[0, 1:-1] = maskC[0, :]
maskC2.data[-1, 1:-1] = maskC[-1, :]
maskC2.data[1:-1, 0] = maskC[:, 0]
maskC2.data[1:-1, -1] = maskC[:, -1]
# remove single land point in Atlantic corner
maskC2.data[-1, 0] = 1

Manual removing of unwanted cells

In [ ]:
bmask = maskC2.copy().values.T[:, :]
# Create the figure
fig, ax = plt.subplots(figsize=(11, 10))
ax.set_title('Click and drag to change the mask.\nLeft button inverts the mask\nRight button fills the selected region')
im = ax.imshow(bmask, cmap=mpl.cm.bone, alpha=.5, origin="lower")

def onselect(click, release):
    """With left button, reverse the value of the mask for all elements inside the rectangle
    formed by the click, drag and release of the mouse.
    
    With the right button, fill all elements inside the rectangle with the value it takes 
    at the click corner.
    """
    x1, y1 = click.xdata, click.ydata
    x2, y2 = release.xdata, release.ydata       
    signx = int(np.sign(x2 - x1 - 0.1))
    signy = int(np.sign(y2 - y1 - 0.1))
    sx = slice(int(x1+1), int(x2)+signx, signx)
    sy = slice(int(y1+1), int(y2)+signy, signy)
    if click.button == 1:
        bmask[sy, sx] = 1
    elif click.button == 3:
        bmask[sy, sx] = 0
    im.set_data(bmask)
    fig.canvas.draw_idle()

# Connect 'click and drag' events to method 'onselect'.
selector = RectangleSelector(ax, onselect, useblit=True, interactive=True)
plt.show()

In [21]:
l = " " + "\n ".join(["".join(row.astype(str)) for row in bmask[:, :]])

In [22]:
with open("/home/jan/Data/SIM_stuff/mask_dx" + f"{dx:.2f}" + "_nx" + str(nx) + "_ny" + str(ny) + ".dat", "w") as m:
    m.write(l)

In [76]:
for dx_new in [16.0, 8.0, 4.0, 2.0, 1.0]:
    if dx_new == 16.0:
        nx_new = 320
        ny_new = 270
    elif dx_new == 8.0:
        nx_new = 640
        ny_new = 540
    elif dx_new == 4.0:
        nx_new = 1280
        ny_new = 1080
    elif dx_new == 2.0:
        nx_new = 2560
        ny_new = 2160
    elif dx_new == 1.0:
        nx_new = 5120
        ny_new = 4320
    long_new, lat_new = create_long_lat(dx_new, nx_new+2, ny_new+2, 32.0)
    mask_new = xr.DataArray(np.zeros((nx_new+2, ny_new+2), dtype=int), 
                             dims={"x": np.arange(0, nx_new+2), "y": np.arange(0, ny_new+2)}, 
                             coords={"lon": (["x", "y"], long_new), "lat": (["x", "y"], lat_new)})
    mask_new.data[1:-1, 1:-1] = mask32.interp(x=np.arange(-dx_new/32/2, nx_new/(32/dx_new)-(dx_new/32/2), dx_new/32), 
                                              y=np.arange(-dx_new/32/2, ny_new/(32/dx_new)-(dx_new/32/2), dx_new/32), 
                                              method="nearest", kwargs={"fill_value": "extrapolate"}).values
    # now adding the halos
    mask_new.data[0, 2:-2] = mask_new.data[1, 2:-2]
    mask_new.data[-1, 2:-2] = mask_new.data[-2, 2:-2]
    mask_new.data[2:-2, 0] = mask_new.data[2:-2, 1]
    mask_new.data[2:-2, -1] = mask_new.data[2:-2, -2]
    mask_new.data[-2::, 0:2] = 1 # fill the Atlantic corner
    nmask = mask_new.values.T
    l = " " + "\n ".join(["".join(row.astype(str)) for row in nmask[:, :]])
    with open("/home/jan/Data/SIM_stuff/mask_dx" + f"{dx_new:.2f}" + "_nx" + str(nx_new) + "_ny" + str(ny_new) + "_base32.dat", "w") as m:
        m.write(l)
    # make sure all land values are -10 in bathymetry file
    with open("/home/jan/Data/SIM_stuff/bathymetry_dx" + f"{dx_new:.2f}" + "_nx" + str(nx_new) + "_ny" + str(ny_new) + ".dat", "r") as b:
        reader = csv.reader(b, delimiter="\t")
        bathy_new = np.array([row for row in reader]) 
    bathy_new = np.where(mask_new.values==0, -10.0, bathy_new)
    m = " " + "\n ".join(["".join(row.astype(str)) for row in bathy_new.T[:, :]])
    with open("/home/jan/Data/SIM_stuff/bathymetry_dx" + f"{dx_new:.2f}" + "_nx" + str(nx_new) + "_ny" + str(ny_new) + "_base32.dat", "w") as b:
        b.write(m)